# "Find me more like this one": similarity search in the FLARE feature space

### Teaching notebook 2, a companion to `flare_tutorial.ipynb`

In the first notebook we taught a machine to *classify* light curves into six classes
(SN Ia, core-collapse supernovae, superluminous supernovae, AGN, tidal disruption events
and cataclysmic variables). Astronomers often ask a different question:

> *"I found this strange object. Are there **others like it** in the archive?"*

That question is **similarity search**, and at ZTF and Rubin scale (millions of alerts per
night) nobody can answer it by eye. In this lesson we build a similarity-search engine on
top of FLARE, using its **160 light-curve features as an embedding**: every object becomes
a point in a 160-dimensional space, and *"similar objects"* simply means *"nearby points"*.

**Why the FLARE features make a good embedding.** Unlike the hidden vector of a neural
network, every dimension has a physical meaning (a colour, a timescale, a smoothness).
When two objects are neighbours, we can say **why**.

**The plan:**
1. Turn the archive into vectors
2. Exact nearest-neighbour search: query with a TDE
3. A 2-D map of the transient sky
4. k-NN as a classifier, and how it compares to the trees
5. Distance as a **weirdness score**: anomaly detection
6. "Find more like this one" for objects outside the six classes: novae retrieving novae
7. Scaling to millions: approximate search

> This notebook is a compact, real-data companion to the full lecture series
> [`SimilaritySearch`](https://github.com/asasli/SimilaritySearch), which goes
> deeper into metric learning, multimodal retrieval and active-learning loops.

> **Requirements:** the `flare` package (`pip install -e ..` from this folder's parent),
> `scikit-learn` and `matplotlib`. All data come from the BTS benchmark shipped in the
> repository (`benchmark/`); run `python -c "from flare import benchmark; benchmark.prepare()"`
> once if `benchmark/events/` does not exist yet.

In [ ]:
# --- Setup ---------------------------------------------------------------
import sys, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "..")                 # find the flare package if not pip-installed
import flare
from flare import benchmark as B
from flare import data as D

if not (B.BENCHMARK / "events").exists():
    B.prepare()                          # unpack the light curves once

CLASSES = B.SIX                          # ["SN_Ia", "SN_CC", "SLSN", "AGN", "TDE", "CV"]
# one fixed colour per class: six evenly spaced shades of magma, avoiding the near-white end
CLASS_COLOR = dict(zip(CLASSES, plt.cm.magma(np.linspace(0.08, 0.82, 6))))
HOLDOUT_COLOR = "0.6"                    # grey for objects outside the six classes
BAND_COLOR = {"g": "seagreen", "r": "crimson", "i": "goldenrod"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

fig, ax = plt.subplots(figsize=(6, 0.6))
for j, c in enumerate(CLASSES):
    ax.add_patch(plt.Rectangle((j, 0), 1, 1, color=CLASS_COLOR[c]))
    ax.text(j + 0.5, 0.5, c, ha="center", va="center", fontsize=9,
            color="white" if j < 3 else "black")
ax.set_xlim(0, 6); ax.set_ylim(0, 1); ax.axis("off"); ax.set_title("class colours", fontsize=9)
plt.show()

## 1. The archive as a cloud of points

Our **archive** is the training and validation part of the benchmark: spectroscopically
classified ZTF objects whose labels we know. Our **queries** are the test objects, which
the shipped FLARE classifier has never seen. The benchmark already contains the 160
light-curve features of every object, computed by the same function the classifier uses,
so we only need to load them.

Two practical details every real system needs:

* **Missing values.** Some objects have no *i*-band and some fits fail, which gives `NaN`.
  Trees handle `NaN` natively, but *distances* do not. We fill each missing value with the
  feature's **median** in the archive.
* **Scaling.** A feature measured in days must not drown one measured in magnitudes. We use
  a **robust scaler** (median and interquartile range), which ignores the extreme outliers
  that astronomical data always contain.

Both are fitted on the archive only and then applied to the queries, exactly as a live
system would treat a new alert.

In [ ]:
# the 160 light-curve features (the benchmark also carries host and catalogue context,
# which we leave out: the embedding is the light curve alone)
FEATS = [c for c in pd.read_parquet(B.BENCHMARK / "features_all.parquet").columns if c != "obj_id"]

parts = {s: B.load_split(s) for s in ("train", "val", "test", "anomaly")}
X_arc_raw = pd.concat([parts["train"][0], parts["val"][0]], ignore_index=True)
y_arc = np.concatenate([parts["train"][1], parts["val"][1]])
t_arc = pd.concat([parts["train"][2], parts["val"][2]], ignore_index=True)
X_test_raw, y_test, t_test = parts["test"]

lab_arc, lab_test = np.array(CLASSES)[y_arc], np.array(CLASSES)[y_test]
print(f"archive: {len(X_arc_raw)} objects x {len(FEATS)} features "
      f"| missing values: {np.isnan(X_arc_raw[FEATS].values).mean():.1%}")
print(f"queries: {len(X_test_raw)} test objects")
pd.DataFrame({"archive": pd.Series(lab_arc).value_counts(),
              "test": pd.Series(lab_test).value_counts()}).loc[CLASSES]

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

imputer = SimpleImputer(strategy="median").fit(X_arc_raw[FEATS])
scaler = RobustScaler().fit(imputer.transform(X_arc_raw[FEATS]))

def embed(X_raw):
    # median-fill, robust-scale and clip the few extreme outliers
    return np.clip(scaler.transform(imputer.transform(X_raw[FEATS])), -10, 10)

E_arc, E_test = embed(X_arc_raw), embed(X_test_raw)
print("embedded:", E_arc.shape, E_test.shape, "(every object is now a point in 160-D space)")

## 2. Query the archive: *"more like this TDE, please"*

The simplest search is **exact k-nearest neighbours**: compute the distance from the query
to every archive point and keep the closest *k*. For a few thousand objects this takes
milliseconds.

We use **cosine distance** (the angle between two vectors), a standard choice for
embeddings, because it compares the *pattern* of features rather than their overall size.

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=5, metric="cosine").fit(E_arc)

# query with a real TDE from the test set (an object the archive does not contain)
q_idx = np.where(lab_test == "TDE")[0][0]
dist, ind = nn.kneighbors(E_test[q_idx:q_idx + 1])
print(f"query: {t_test.obj_id[q_idx]} (a real TDE)\n")
print("rank  object          class   cosine distance")
for r, (i, d) in enumerate(zip(ind[0], dist[0]), start=1):
    print(f"  {r}   {t_arc.obj_id[i]:14s}  {lab_arc[i]:6s}  {d:.3f}")

In [ ]:
def plot_lc(ax, filepath, title, color):
    bands = D.reconstruct_bands(D.load_events(str(B.BENCHMARK / filepath)))
    for name, bc in BAND_COLOR.items():
        if name in bands:
            ax.plot(bands[name].t, bands[name].logflux, "o", ms=3, color=bc, alpha=0.8, label=name)
    ax.set_title(title, color=color, fontsize=10)
    ax.set_xlabel("days")

# the query and its 5 nearest neighbours
fig, axes = plt.subplots(1, 6, figsize=(18, 2.8))
plot_lc(axes[0], t_test.filepath[q_idx], "QUERY: TDE", CLASS_COLOR["TDE"])
for ax, i, d in zip(axes[1:], ind[0], dist[0]):
    plot_lc(ax, t_arc.filepath[i], f"{lab_arc[i]}  d={d:.3f}", CLASS_COLOR[lab_arc[i]])
axes[0].set_ylabel(r"$\log_{10}$ flux"); axes[0].legend(fontsize=7)
plt.suptitle("The query TDE (left) and its 5 nearest neighbours in feature space", y=1.06)
plt.tight_layout(); plt.show()

The nearest neighbour of our TDE is **another TDE** (cosine distance 0.262), even though
the archive holds only 58 TDEs among 8922 objects. The next four are three SN Ia and one
core-collapse supernova with the same smooth rise and slow fall. TDE-like supernovae are
precisely the confusion that matters scientifically, and because the dimensions are
physical, we can ask **which features made them neighbours** (colour, rise time, decline
rate, smoothness).

### Exercise 1
Query with a **CV** instead (change `"TDE"` to `"CV"` above). Are its neighbours also CVs?
Why do objects with repeated outbursts cluster so tightly? Then try an **SLSN**: does the
search return superluminous supernovae, or ordinary core-collapse ones?

## 3. A map of the transient sky

160 dimensions are hard to imagine. We project the archive to 2-D with **PCA** (keeping
the two directions of largest variance) and colour it by true class, which gives a map
where similar objects sit together. SN Ia make up most of the archive, so we draw them
first and the rarer classes on top.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=0).fit(E_arc)
Z_arc, Z_test = pca.transform(E_arc), pca.transform(E_test)
fig, ax = plt.subplots(figsize=(9, 5.5))
for cls in CLASSES:
    m = lab_arc == cls
    ax.scatter(Z_arc[m, 0], Z_arc[m, 1], s=6 if cls == "SN_Ia" else 12,
               alpha=0.35 if cls == "SN_Ia" else 0.8, color=CLASS_COLOR[cls],
               label=f"{cls} (n={m.sum()})")
ax.scatter(Z_test[q_idx, 0], Z_test[q_idx, 1], s=120, facecolor="none",
           edgecolor="black", lw=2, label="our TDE query")
ax.set_xlabel("PCA 1"); ax.set_ylabel("PCA 2")
ax.set_title("The archive, embedded and flattened to 2-D")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()
print(f"variance kept by the two PCA axes: {pca.explained_variance_ratio_.sum():.1%}")

The two PCA axes keep 32.3% of the variance, so the map is a rough sketch of the
160-D space. Even so, the classes do not mix uniformly: many CVs spread far along the
second axis, away from the dense regions that the supernovae fill. The embedding organises
the sky by physics without being told the labels. (For prettier maps, try UMAP or t-SNE;
PCA is the honest, linear baseline.)

## 4. k-NN as a classifier

If most of my neighbours are AGN, I am probably an AGN. That is **k-NN classification**:
no training at all, just the archive and a distance. We test it honestly by letting every
test object vote with its *k* nearest archive neighbours, and we compare with the shipped
FLARE classifier on the same test objects (it was trained on the same archive).

With six classes this unbalanced, plain accuracy mostly measures how well we recognise
SN Ia, so we also report the **balanced accuracy** (the mean recall over the six classes).

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score

k = 15
_, ind_test = NearestNeighbors(n_neighbors=k, metric="cosine").fit(E_arc).kneighbors(E_test)
votes = y_arc[ind_test]                                          # (N_test, k) class indices
knn_pred = np.array([np.bincount(v, minlength=6).argmax() for v in votes])

clf = flare.load_classifier()                                    # six-class FLARE
flare_pred = np.array([CLASSES.index(c) for c in clf.predict(X_test_raw.reindex(columns=clf.feature_names))])

rows = {}
for name, pred in [(f"k-NN (k={k})", knn_pred), ("FLARE (trees)", flare_pred)]:
    rows[name] = {"accuracy": accuracy_score(y_test, pred),
                  "balanced accuracy": balanced_accuracy_score(y_test, pred),
                  **dict(zip([f"recall {c}" for c in CLASSES],
                             recall_score(y_test, pred, labels=range(6), average=None)))}
pd.DataFrame(rows).T.round(2)

**Lesson.** Plain neighbourhood voting reaches an accuracy of 0.83 against 0.93 for FLARE,
and both recover 96% of the SN Ia. The gap opens in the rare classes: the balanced accuracy
is 0.44 for k-NN and 0.84 for FLARE, and k-NN finds only 17% of the SLSNe and 1 of the 10
test TDEs (recall 0.1). Two thirds of the archive are SN Ia, so a majority vote among 15
neighbours rarely lets a rare class win. The boosted trees learn *which* features matter
for *which* boundary, weight the rare classes, and also read host-galaxy and catalogue
context that our light-curve embedding leaves out. k-NN still needs **zero training** and
gives retrieval for free, so in production the two complement each other: FLARE for the
probabilities, the index for "show me similar ones".

### Exercise 2
Try `k = 3` and `k = 50`. Which classes benefit from a small `k`, which from a large one,
and why? (Hint: how many TDEs does the whole archive contain, and how many SN Ia?) Then
weight each vote by the inverse of the class frequency in the archive and watch the
balanced accuracy.

## 5. Distance as a weirdness score

We now flip the logic: an object whose nearest neighbours are **far away** is *unlike
everything in the archive*, which makes it an **anomaly**. At survey scale this is how we
hunt for objects nobody has seen before. Our weirdness score is the distance to the
*k*-th nearest archive neighbour.

Here the choice of distance matters. Cosine distance compares only the *pattern* of the
features, which suits retrieval, but it ignores *how far* an object lies from the typical
one. An object with the right pattern and extreme values (a huge amplitude, a very fast
decline) looks ordinary to cosine distance. For weirdness we therefore use the
**Euclidean distance**, which counts both, and we test that choice below.

In [ ]:
knn_euc = NearestNeighbors(n_neighbors=k).fit(E_arc)              # Euclidean distance
weird_test = knn_euc.kneighbors(E_test)[0][:, -1]
order = np.argsort(weird_test)[::-1]

fig, axes = plt.subplots(1, 5, figsize=(16, 2.8))
for ax, i in zip(axes, order[:5]):
    plot_lc(ax, t_test.filepath[i], f"{lab_test[i]}  d={weird_test[i]:.1f}", CLASS_COLOR[lab_test[i]])
axes[0].set_ylabel(r"$\log_{10}$ flux")
plt.suptitle("The 5 'weirdest' test objects (largest k-NN distance to the archive)", y=1.06)
plt.tight_layout(); plt.show()

top = pd.Series(lab_test[order[:40]]).value_counts().reindex(CLASSES, fill_value=0)
base = pd.Series(lab_test).value_counts(normalize=True).reindex(CLASSES)
pd.DataFrame({"in top-40 weirdest": top, "expected from class mix": (40 * base).round(1)})

The five weirdest test objects are four core-collapse supernovae and one SN Ia, all with
long, densely sampled light curves that keep evolving for 100 days, far from the typical
archive object. Among the 40 weirdest, the rare and slow classes are over-represented:
16 SN_CC (9.8 expected from the class mix), 3 SLSNe (0.6 expected) and 4 CVs (1.1
expected), while SN Ia, the most common class, drop to 16 (27.3 expected). A human can
look at 40 candidates per night but not at 500,000. **That is the point of similarity
search.**

The real test is the benchmark's **out-of-taxonomy hold-out**: 53 spectroscopically
classified objects that belong to *none* of the six classes (novae, luminous blue
variables, luminous red novae, intermediate-luminosity red transients, calcium-rich
transients and a fast blue optical transient). The archive contains none of them. If
distance measures weirdness, these objects should sit farther from the archive than the
ordinary test objects do.

In [ ]:
from sklearn.metrics import roc_auc_score

X_ho_raw, _, t_ho = parts["anomaly"]
E_ho = embed(X_ho_raw)
knn_cos = NearestNeighbors(n_neighbors=k, metric="cosine").fit(E_arc)
is_nova = t_ho.bts_type.values == "nova"

rows = {}
for name, index in [("cosine", knn_cos), ("Euclidean", knn_euc)]:
    wt, wh = index.kneighbors(E_test)[0][:, -1], index.kneighbors(E_ho)[0][:, -1]
    auc = lambda w: roc_auc_score(np.r_[np.zeros(len(wt)), np.ones(len(w))], np.r_[wt, w])
    rows[name] = {"AUROC, whole hold-out": auc(wh), "AUROC, novae only": auc(wh[is_nova])}
print("How well does weirdness separate the hold-out from the test objects? (0.5 = chance)")
display(pd.DataFrame(rows).T.round(3))

weird_ho = knn_euc.kneighbors(E_ho)[0][:, -1]
fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.linspace(0, max(weird_test.max(), weird_ho.max()), 40)
ax.hist(weird_test, bins=bins, density=True, alpha=0.6, color=CLASS_COLOR["SN_CC"], label="test (six classes)")
ax.hist(weird_ho, bins=bins, density=True, alpha=0.6, color=HOLDOUT_COLOR, label="out-of-taxonomy hold-out")
ax.set_xlabel(f"Euclidean distance to the {k}-th nearest archive neighbour"); ax.set_ylabel("density")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

fam = t_ho.bts_type.values
pd.DataFrame({"n": pd.Series(fam).value_counts(),
              "median weirdness": pd.Series(weird_ho).groupby(fam).median().round(1),
              "features missing": X_ho_raw[FEATS].isna().mean(1).groupby(fam).mean().round(2)}
             ).sort_values("n", ascending=False)

The choice of distance decides the result. With cosine distance the weirdness barely
separates the hold-out from the test objects (AUROC 0.573), while Euclidean distance reaches
0.756 for the whole hold-out and 0.810 for the 37 novae. In the histogram the hold-out sits
at larger distances than most test objects, although the two distributions overlap.

The family table also shows a blind spot. The single fast blue optical transient (FBOT) has
**all** its features missing (fraction 1.00), so the median filling places it exactly at the
centre of the archive, and its weirdness of 3.4 is lower than that of every other hold-out
family. An object we cannot measure looks perfectly ordinary to a distance, so a real
system must flag missing data separately.

### Exercise 3
Plot the histogram of `weird_test` per class (`plt.hist` with `CLASS_COLOR`). Which class is
intrinsically the most "self-similar"? Then compare with the classifier's own anomaly score,
`clf.anomaly_energy(X_ho_raw.reindex(columns=clf.feature_names))`.

## 6. "Find more like this one": novae retrieving novae

Anomaly detection hands a human a short list of strange objects. Once the human recognises
one of them, say a **nova**, the next question is *"where are the others?"*. Similarity
search answers it without any retraining: we add the hold-out objects to the index and
query with one nova. A random draw from this index would return a nova only rarely, so
every nova among the neighbours is a real gain.

In [ ]:
# the index now holds the archive plus the hold-out; the labels of the hold-out
# are its spectroscopic families
E_idx = np.vstack([E_arc, E_ho])
lab_idx = np.r_[lab_arc, t_ho.bts_type.values]
fp_idx = np.r_[t_arc.filepath.values, t_ho.filepath.values]
id_idx = np.r_[t_arc.obj_id.values, t_ho.obj_id.values]
n_arc = len(E_arc)
color_of = lambda lab: CLASS_COLOR.get(lab, HOLDOUT_COLOR)

nn_idx = NearestNeighbors(n_neighbors=6, metric="cosine").fit(E_idx)
_, ind_idx = nn_idx.kneighbors(E_idx[n_arc:])                  # every hold-out object as a query
ind_idx = ind_idx[:, 1:]                                       # drop the query itself

n_nova = is_nova.sum()
prec = (lab_idx[ind_idx[is_nova]] == "nova").mean()
print(f"{n_nova} novae in an index of {len(E_idx)} objects: "
      f"a random neighbour is a nova with probability {(n_nova - 1) / (len(E_idx) - 1):.4f}")
print(f"fraction of novae among the 5 nearest neighbours of a nova: {prec:.2f}")
print(f"novae with at least one nova among their 5 neighbours: "
      f"{(lab_idx[ind_idx[is_nova]] == 'nova').any(1).sum()} of {n_nova}")

# which classes take the other places?
pd.Series(lab_idx[ind_idx[is_nova]].ravel()).value_counts().rename("neighbours of novae").to_frame().T

In [ ]:
# look at one example: the nova whose neighbours contain the most novae
q = np.where(is_nova)[0][np.argmax((lab_idx[ind_idx[is_nova]] == "nova").sum(1))]
d_q, i_q = nn_idx.kneighbors(E_idx[n_arc + q:n_arc + q + 1])
fig, axes = plt.subplots(1, 6, figsize=(18, 2.8))
for r, (ax, i, d) in enumerate(zip(axes, i_q[0], d_q[0])):
    title = f"QUERY: {lab_idx[i]}" if r == 0 else f"{lab_idx[i]}  d={d:.3f}"
    plot_lc(ax, fp_idx[i], title, color_of(lab_idx[i]))
axes[0].set_ylabel(r"$\log_{10}$ flux"); axes[0].legend(fontsize=7)
plt.suptitle(f"A nova ({id_idx[n_arc + q]}, left) and its 5 nearest neighbours", y=1.06)
plt.tight_layout(); plt.show()

With 37 novae in an index of 8975 objects, a random neighbour is a nova with probability
0.0040. Among the five nearest neighbours of a nova, the fraction of novae is 0.07, more
than 15 times the random rate, and 13 of the 37 novae have at least one nova among their
five neighbours. The index uses no labels at all, yet the features place
novae near each other. The other places go mostly to SN Ia (79), CVs (43) and SN_CC
(36). The CVs are a physically sensible confusion, because a nova is also an eruption on a
white dwarf that accretes from a companion star.

This is how similarity search supports discovery: anomaly detection proposes the first
member of a new family, a human recognises it, and retrieval returns a short, enriched
list of candidate siblings to inspect next.

## 7. Scaling up: approximate nearest neighbours

Exact search compares the query with *every* archive point. That is fine for $10^4$ objects
and impossible for $10^9$. **Approximate nearest-neighbour** methods build an index that
looks at only a small part of the archive for each query, trading a little recall for a
large gain in speed.

The idea fits in a few lines of scikit-learn. An **inverted-file index** first splits the
archive into clusters with k-means; a query then searches only the points of its
`n_probe` closest clusters. We measure the **recall@5** (the fraction of the true five
nearest neighbours that the approximate search also finds) and the time per query.
Dedicated libraries (FAISS, Annoy, Voyager) implement the same idea, and much more, in
optimised code; the cell uses them if they are installed. The full comparison lives in the
[`SimilaritySearch`](https://github.com/asasli/SimilaritySearch) lecture series.

In [ ]:
from sklearn.cluster import KMeans

# Euclidean search on unit-length vectors ranks neighbours exactly as cosine distance does
unit = lambda E: (E / np.linalg.norm(E, axis=1, keepdims=True)).astype(np.float32)
A, Q = unit(E_arc), unit(E_test)

t0 = time.perf_counter()
_, exact = NearestNeighbors(n_neighbors=5, algorithm="brute").fit(A).kneighbors(Q)
t_exact = (time.perf_counter() - t0) / len(Q)

n_clusters = 64
km = KMeans(n_clusters=n_clusters, n_init=4, random_state=0).fit(A)
members = [np.where(km.labels_ == c)[0] for c in range(n_clusters)]

def ivf_search(qv, n_probe, k=5):
    near = np.argsort(((km.cluster_centers_ - qv) ** 2).sum(1))[:n_probe]
    cand = np.concatenate([members[c] for c in near])
    return cand[np.argsort(((A[cand] - qv) ** 2).sum(1))[:k]]

print(f"exact (scikit-learn, brute force): {1e3 * t_exact:.3f} ms / query, recall@5 = 1.000")
for n_probe in (1, 4, 16):
    t0 = time.perf_counter()
    approx = np.array([ivf_search(qv, n_probe) for qv in Q])
    dt = (time.perf_counter() - t0) / len(Q)
    recall = np.mean([len(set(a) & set(e)) / 5 for a, e in zip(approx, exact)])
    print(f"IVF, {n_probe:2d} of {n_clusters} clusters  : {1e3 * dt:.3f} ms / query, "
          f"recall@5 = {recall:.3f}, compares with ~{n_probe / n_clusters:.0%} of the archive")

try:
    import faiss
    index = faiss.IndexFlatL2(A.shape[1]); index.add(A)
    t0 = time.perf_counter(); index.search(Q, 5)
    print(f"FAISS flat: {1e3 * (time.perf_counter() - t0) / len(Q):.3f} ms / query")
except ImportError:
    print("FAISS is not installed (`pip install faiss-cpu` to try it)")
try:
    from annoy import AnnoyIndex
    ann = AnnoyIndex(A.shape[1], "euclidean")
    for i, v in enumerate(A):
        ann.add_item(i, v)
    ann.build(20)
    t0 = time.perf_counter()
    for qv in Q:
        ann.get_nns_by_vector(qv, 5)
    print(f"Annoy (20 trees): {1e3 * (time.perf_counter() - t0) / len(Q):.3f} ms / query")
except ImportError:
    print("Annoy is not installed (`pip install annoy` to try it)")

With 4 of the 64 clusters (about 6% of the archive per query) the inverted file already
recovers 94.0% of the true five nearest neighbours, and with 16 clusters it recovers 99.7%.
On an archive of only 8922 objects the exact search remains faster, because scikit-learn
compares all queries with all points in one vectorised step while our index loops in
Python. The gain of the approximate index grows with the size of the archive: its cost
scales with the number of points it inspects, which stays a small fraction of the
archive.

## 8. What we learned

1. A **feature vector is an embedding**, and physics features give an *interpretable* one.
2. **k-NN retrieval** answers "find more like this" in milliseconds, and we can explain the
   neighbours feature by feature.
3. A 2-D **map** of the embedding shows the classes arranging themselves by physics alone.
4. **k-NN classification** is a zero-training baseline; boosted trees refine it, above all
   for the rare classes. A classifier plus an index makes a complete production system.
5. **Distance measures weirdness**: the same index is an anomaly detector at no extra cost,
   provided we measure how far, not only in which direction (Euclidean rather than cosine),
   and flag objects whose features are missing.
6. Once a human recognises one new object, the index **retrieves its siblings** without
   any retraining.
7. **Approximate search** (inverted files, FAISS, Annoy) makes all of this work at Rubin
   scale.

### Final challenge
Build the index from **day-10 features** instead. The benchmark ships the 100-day features,
so we recompute them for early times:
`clf.features_from_files([B.BENCHMARK / p for p in t_arc.filepath], horizon_days=10)[FEATS]`
(and the same for the test objects; expect a few minutes). Query with the same TDE. Are its
early-time neighbours still TDE-like, or do young TDEs hide among the supernovae? What does
that mean for real-time follow-up?

---
*Continue with the full lecture series,
[github.com/asasli/SimilaritySearch](https://github.com/asasli/SimilaritySearch):
metric learning (make TDEs cluster tighter), multimodal retrieval (light curve to
spectrum) and active-learning feedback loops.*